# Deep learning project

In [1]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader, WeightedRandomSampler, ConcatDataset
import segmentation_models_pytorch as smp

from tqdm import tqdm
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

import os
from PIL import Image
from road_dataset import RoadImageDataset, train_transform, val_transform

# Shapes are fixed (512x512 crops/windows, drop_last=True on train), so the
# autotuner's one-time search pays for itself within the first few steps.
torch.backends.cudnn.benchmark = True

root_massachusetts = r"C:\Dev\deeplearning\data\massachusetts-roads-dataset"
root_deepglobe = r"C:\Dev\deeplearning\data\deepglobe"


def _window_positions(size, window, stride):
    """Top-left offsets covering `size` with `window`-sized steps of `stride`,
    with a final offset snapped to the edge so the last window is flush."""
    if size <= window:
        return [0]
    positions = list(range(0, size - window + 1, stride))
    if positions[-1] != size - window:
        positions.append(size - window)
    return positions


@torch.no_grad()
def sliding_window_predict(model, images, window=512, stride=384, max_batch=32):
    """Tile `images` (B,C,H,W) into overlapping windows, run the model on each,
    and stitch the probabilities back onto a full-size canvas, averaging where
    windows overlap. Returns (B,1,H,W) probabilities at full resolution."""
    B, C, H, W = images.shape
    coords = [
        (y, x)
        for y in _window_positions(H, window, stride)
        for x in _window_positions(W, window, stride)
    ]

    # (num_windows * B, C, window, window), grouped window-major
    batched = torch.cat([images[:, :, y:y + window, x:x + window] for y, x in coords], dim=0)

    probs_chunks = []
    for i in range(0, batched.size(0), max_batch):
        chunk = batched[i:i + max_batch]
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            logits = model(chunk)
        probs_chunks.append(torch.sigmoid(logits.float()))
    probs = torch.cat(probs_chunks, dim=0).view(len(coords), B, 1, window, window)

    prob_sum = torch.zeros((B, 1, H, W), device=images.device)
    count = torch.zeros((B, 1, H, W), device=images.device)
    for i, (y, x) in enumerate(coords):
        prob_sum[:, :, y:y + window, x:x + window] += probs[i]
        count[:, :, y:y + window, x:x + window] += 1

    return prob_sum / count.clamp(min=1)


# Windows requires this block for multiprocessing
if __name__ == "__main__":
    print("[+] loading datasets...")

    SEED = 42
    mass_train = RoadImageDataset(root_massachusetts, "train", train_transform)
    mass_val = RoadImageDataset(root_massachusetts, "val", val_transform)

    dg_train = RoadImageDataset(
        root_deepglobe, "train", train_transform, val_fraction=0.1, seed=SEED
    )
    dg_val = RoadImageDataset(
        root_deepglobe, "val", val_transform, val_fraction=0.1, seed=SEED
    )

    train_dataset = ConcatDataset([mass_train, dg_train])
    print(
        f"[+] train: {len(mass_train)} mass + {len(dg_train)} dg = {len(train_dataset)}"
    )

    val_dataset = ConcatDataset([mass_val, dg_val])

    # Define per-sample weights
    weight_mass = 1.0 / len(mass_train)
    weight_dg = 1.0 / len(dg_train)

    # Build the combined weight list
    weights = [weight_mass] * len(mass_train) + [weight_dg] * len(dg_train)

    # Create the sampler (replacement=True is mandatory for WeightedRandomSampler)
    sampler = WeightedRandomSampler(
        weights=weights, num_samples=len(train_dataset), replacement=True
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=8,
        sampler=sampler,  # Use sampler here instead of shuffle=True
        num_workers=4,
        pin_memory=True,
        persistent_workers=True,
        prefetch_factor=4,
        drop_last=True,
    )

    # val tiles are full resolution (no crop) and vary in size between datasets,
    # so each loader must stay single-dataset for the default collate to stack them.
    # Lighter worker pool than train: two of these loaders idling with
    # persistent_workers=True would add 8 more live processes for no benefit —
    # they only get iterated once per epoch, not worth keeping warm.
    val_loaders = {
        "mass": DataLoader(
            mass_val,
            batch_size=8,
            shuffle=False,
            num_workers=2,
            pin_memory=True,
            persistent_workers=False,
        ),
        "dg": DataLoader(
            dg_val,
            batch_size=8,
            shuffle=False,
            num_workers=2,
            pin_memory=True,
            persistent_workers=False,
        ),
    }

    # Model definition
    model = smp.Unet(
        encoder_name="resnet34",
        encoder_weights="imagenet",
        in_channels=3,
        classes=1,
    )

    # Pre-training initialization
    print("[+] started initializations...")

    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available")

    device = torch.device("cuda")
    model = model.to(device, memory_format=torch.channels_last)
    # Compiled once here; used for both the fixed-shape 512x512 train step and
    # the sliding-window val forward passes (Dynamo falls back to per-shape
    # recompiles for the latter's varying last-chunk size, then stabilizes).
    model = torch.compile(model)

    # ToDtype(float32, scale=True) + Normalize, done on-GPU instead of in the worker.
    # The dataset now hands over uint8 (0.75MB/512x512x3 sample instead of 3MB),
    # so this is 4x less PCIe traffic and less CPU work per sample.
    IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
    IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    def to_gpu_input(images):
        images = images.to(device, non_blocking=True, memory_format=torch.channels_last)
        images = images.float().div_(255.0)
        return images.sub_(IMAGENET_MEAN).div_(IMAGENET_STD)

    criterion = smp.losses.DiceLoss(mode="binary", from_logits=True)
    # Validation loss is computed on the merged, full-tile probability map (not per-window
    # logits), so it takes probabilities directly instead of converting from logits itself.
    val_criterion = smp.losses.DiceLoss(mode="binary", from_logits=False)
    optimizer = optim.Adam(model.parameters(), lr=1e-4, fused=True)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=2
    )

    print("[+] starting the training...")
    num_epochs = 20
    best_val = float("inf")
    ckpt_dir = r"C:\Dev\deeplearning\trained_models"

    print("[+] starting epoch...")
    for epoch in range(num_epochs):
        model.train()
        running_loss = torch.zeros((), device=device)

        print("starting batching...")
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            images = to_gpu_input(images)
            labels = labels.to(device, non_blocking=True).float()

            optimizer.zero_grad(set_to_none=True)

            with torch.amp.autocast("cuda", dtype=torch.bfloat16):
                outputs = model(images)

            # Loss computed on fp32 logits
            loss = criterion(outputs.float(), labels)

            loss.backward()
            optimizer.step()

            # Accumulate on-GPU — .item() every step would sync the host to the
            # device and stall the pipeline behind backward(). Read it once below.
            running_loss += loss.detach()

        avg_train_loss = (running_loss / len(train_loader)).item()
        print(f"Epoch {epoch+1}/{num_epochs} - Train Loss: {avg_train_loss:.4f}")

        # --- validation: merge predictions across overlapping windows into one
        # full-resolution mask per tile, THEN accumulate the confusion matrix and
        # loss globally over the whole val set. No per-window/per-crop metrics.
        model.eval()
        val_losses = {}

        with torch.no_grad():
            for name, loader in val_loaders.items():
                total_loss = torch.tensor(0.0, device=device)
                n_tiles = 0
                tp = torch.tensor(0, device=device)
                fp = torch.tensor(0, device=device)
                fn = torch.tensor(0, device=device)

                for images, labels in loader:
                    images = to_gpu_input(images)
                    labels = labels.to(device, non_blocking=True).float()

                    probs = sliding_window_predict(model, images, window=512, stride=384)

                    total_loss += val_criterion(probs, labels) * images.size(0)
                    n_tiles += images.size(0)

                    pred = probs > 0.5
                    tgt = labels.bool()
                    tp += (pred & tgt).sum()
                    fp += (pred & ~tgt).sum()
                    fn += (~pred & tgt).sum()

                # Single sync per loader, not per batch.
                val_losses[name] = (total_loss / n_tiles).item()
                tp_f, fp_f, fn_f = tp.item(), fp.item(), fn.item()
                iou = tp_f / (tp_f + fp_f + fn_f + 1e-7)
                f1 = 2 * tp_f / (2 * tp_f + fp_f + fn_f + 1e-7)
                print(f"  {name}: loss {val_losses[name]:.4f} | IoU {iou:.4f} | F1 {f1:.4f}")

        avg_val_loss = sum(val_losses.values()) / len(val_losses)
        scheduler.step(avg_val_loss)

        if avg_val_loss < best_val:
            best_val = avg_val_loss
            os.makedirs(ckpt_dir, exist_ok=True)
            # Unwrap torch.compile's OptimizedModule before saving so the checkpoint's
            # keys match a plain (uncompiled) model — e.g. what tester.py loads into —
            # instead of being prefixed with "_orig_mod.".
            raw_model = model._orig_mod if hasattr(model, "_orig_mod") else model
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": raw_model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "train_loss": avg_train_loss,
                    "val_loss": avg_val_loss,
                    "val_losses": val_losses,
                },
                os.path.join(ckpt_dir, "roadseg_merged_dice_best.pt"),
            )
            print(f"  [*] new best ({best_val:.4f}) — saved")


[+] loading datasets...
[+] train: 1108 mass + 5604 dg = 6712
[+] started initializations...
[+] starting the training...
[+] starting epoch...
starting batching...


Epoch 1: 100%|██████████| 839/839 [01:27<00:00,  9.55it/s]


Epoch 1/20 - Train Loss: 0.6321
  mass: loss 0.3469 | IoU 0.5218 | F1 0.6858
  dg: loss 0.3999 | IoU 0.4933 | F1 0.6607
  [*] new best (0.3734) — saved
starting batching...


Epoch 2: 100%|██████████| 839/839 [01:14<00:00, 11.24it/s]


Epoch 2/20 - Train Loss: 0.3783
  mass: loss 0.2912 | IoU 0.5591 | F1 0.7172
  dg: loss 0.3467 | IoU 0.5040 | F1 0.6702
  [*] new best (0.3189) — saved
starting batching...


...

Epoch 18/20 - Train Loss: 0.2919
  mass: loss 0.2491 | IoU 0.6025 | F1 0.7519
  dg: loss 0.2696 | IoU 0.5868 | F1 0.7396
starting batching...


Epoch 19: 100%|██████████| 839/839 [01:14<00:00, 11.26it/s]


Epoch 19/20 - Train Loss: 0.2832
  mass: loss 0.2470 | IoU 0.6058 | F1 0.7545
  dg: loss 0.2791 | IoU 0.5744 | F1 0.7297
starting batching...


Epoch 20: 100%|██████████| 839/839 [01:14<00:00, 11.26it/s]


Epoch 20/20 - Train Loss: 0.2782
  mass: loss 0.2460 | IoU 0.6066 | F1 0.7551
  dg: loss 0.2575 | IoU 0.6005 | F1 0.7504
  [*] new best (0.2517) — saved
